# p4a 执行轨迹观测

前缀层面的问题已在 `01_session_classes.ipynb` 完成实验。

本笔记选择 961 份 sessions ，观察和检测 agent 实际走出来的步骤有多少是共通的、从哪里开始分叉。

观测集是那 961 份。选它的理由见 `docs/experiments/e01-p4a-trajectory.md` §1.3：组内前缀已构造性同质，观察到的分叉可以归因于轨迹本身。

## 1. 筛出观测集

判据是三元组 `(工具Δ, 目录树, delivery)`，取 `(-3882, fb389653, pointer)`，再叠加 s0 的纳入过滤。

In [ ]:
import nbio
import pandas as pd
import plotly.express as px

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
nbio.banner()

In [ ]:
# 观测集判据只在 e01/n0_fix_timestamp.py 里定义一次，notebook 不抄。
from e01.n0_fix_timestamp import GROUP

w = nbio.wide()                                   # s0 × s0b × s2 × s3 × s4
sel = w.included.copy()
for k, v in GROUP.items():
    sel &= w[k] == v
g = w[sel].copy()
g["day"] = pd.to_datetime(g.created_at).dt.tz_convert("Asia/Shanghai").dt.date

assert len(g) == 961, f"观测集份数不是 961 而是 {len(g)}，判据或产物变了"
assert g.sysprompt_chars.nunique() == 1, "组内 systemPrompt 长度不唯一"
assert (g.family == "extract").all(), "组内混入了非 extract"

print(GROUP)
print(f"观测集 {len(g)} 份 | {g.paper_id.nunique()} 篇论文 | {g.day.min()} .. {g.day.max()}")
print(f"sysprompt_chars {g.sysprompt_chars.iloc[0]} | 累计 prefill {g.sum_input.sum():,.0f} tok "
      f"（占纳入集 {g.sum_input.sum() / w[w.included].sum_input.sum():.1%}）")

筛完先自查同质性：组内首步 prompt 的跨度就是这一组"前缀有多齐"的直接度量。

In [ ]:
fs = g.first_step_input
print(f"首步 prompt token: min {fs.min()} max {fs.max()}，跨度 {fs.max() - fs.min()} tok")
print(f"时间戳之后被切断的字符 p50: {g.poisoned_tail_chars.median():.0f}")
print()
print("组内仍在变的列（互异 > 1）：")
vary = {c: g[c].nunique() for c in
        ["axis_harness", "axis_tree", "axis_agents", "axis_skills", "axis_timestamp",
         "sysprompt_chars", "tools_tok_rel_ref", "delivery", "pointer_layout", "pid_year"]
        if g[c].nunique() > 1}
print(vary)

只有时间戳在变，其余各轴组内为常数。这正是 §1.3 说的"L2 按构造成立"。

In [ ]:
desc = (g[["n_steps", "n_tools", "peak_input", "sum_input", "amplification",
           "n_external", "n_bash", "n_read", "n_edit", "n_todo", "n_injections"]]
        .describe(percentiles=[.1, .25, .5, .75, .9]).T
        .drop(columns=["count"]).round(1))
desc

## 2. 归一化：固定时间戳

组内只有时间戳一轴在变。把它换成一个常量，systemPrompt 在 961 份之间就逐字相同。

这一步由 `make n0` 产出，判据、常量与自检都在 `e01/n0_fix_timestamp.py`。产物
`data/processed/e01/n0_fixed_time/` 与原始语料同构，逐字节相同，只差 systemPrompt
里那 24 个字符；上下文流只来自 `wire.jsonl`，故只复制它和 `state.json`，53 份带子
agent 的，子 agent wire 同样处理。

流里其余像时间戳的东西是内容不是时钟 —— arXiv 的 `published`、GitHub 的 commit
date、论文的 `submitted` 字段 —— 不动。`state.json` 的 `createdAt` 也不动，它不进
上下文，改了会毁掉时序。

In [ ]:
n0 = nbio.summary("n0")
ck = n0["checks"]
print(f"{n0['corpus']}  {n0['n_sessions']} 份 / {n0['n_papers']} 篇  {n0['corpus_bytes'] / 1e6:.0f} MB")
print(f"时间戳 {n0['orig_timestamp_span']['min']} .. {n0['orig_timestamp_span']['max']}"
      f" -> {n0['fixed_timestamp']}")
print(f"systemPrompt {ck['n_distinct_sysprompt_before']} 个互异值 -> "
      f"{ck['n_distinct_sysprompt_after']} 个（md5 {ck['sysprompt_md5_after'][:8]}）")
print(f"带子 agent 的 session {n0['n_with_subagents']} 份，子 agent wire {n0['n_subagent_wires']} 个")

# 脚本自己选的观测集必须与本笔记筛出的是同一批，否则两边口径已经漂了
ix = nbio.load("n0")
assert set(ix.sid) == set(g.sid), "n0 产物的 sid 集合与本笔记的观测集不一致"
print("✓ 产物与本笔记的观测集是同一批 session")

## 3. 工具使用次数

计数来自 s2 的 `tool_seq`，一次 `tool.call` 算一次，不问成败。

横轴取对数：Read 与最长尾的工具差四个数量级，线性轴上尾巴会全贴在零线。覆盖率
另列一栏 —— 总次数分不清「一份里调 30 次」和「900 份里各调 1 次」。

In [ ]:
calls = g.tool_seq.explode()
tools = pd.DataFrame({
    "调用次数": calls.value_counts(),
    "覆盖 session": g.tool_seq.apply(set).explode().value_counts(),
})
tools.index.name = "tool"
tools["每份均值"] = (tools["调用次数"] / len(g)).round(2)
tools["覆盖率"] = (tools["覆盖 session"] / len(g) * 100).round(1)
tools["来源"] = ["MCP" if t.startswith("mcp__") else "内建" for t in tools.index]
tools = tools.sort_values("调用次数", ascending=False)

fig = px.bar(
    tools.reset_index(), x="调用次数", y="tool", color="来源", orientation="h",
    log_x=True, height=900, text="调用次数",
    title=f"961 份观测集的工具调用次数"
          f"（合计 {tools['调用次数'].sum():,} 次 / {len(tools)} 种）",
    labels={"tool": ""}, hover_data=["每份均值", "覆盖率"],
)
fig.update_yaxes(categoryorder="total ascending")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

tools[["来源", "调用次数", "每份均值", "覆盖 session", "覆盖率"]]

## 4. Skill 正文被读了几次

问题：p4a 的 skill 正文（`skill/paper-mineru-resource-extract/SKILL.md`，460 行 / 21 KB）
是不是够用 —— agent 拿到它之后还要不要回头再读。

判据取「路径以 `SKILL.md` 结尾的 Read」。s2 只留了 `tool_seq` 没留参数，所以路径
得回语料里取；这是探索性扫描，若这条统计要被引用，应当并进 s2 的事件表。

In [ ]:
import json
from collections import Counter

from e01 import wire

CORPUS = nbio.OUT / "n0_fixed_time"

rows, paths = [], Counter()
for d in wire.session_dirs(root=CORPUS):
    reads, globs = [], 0
    for line in open(d / "agents/main/wire.jsonl", encoding="utf-8"):
        if '"tool.call"' not in line:
            continue
        e = json.loads(line).get("event") or {}
        if e.get("type") != "tool.call":
            continue
        a = e.get("args") or {}
        if e.get("name") == "Read" and str(a.get("path", "")).endswith("SKILL.md"):
            reads.append(a["path"])
            paths[a["path"]] += 1
        elif e.get("name") == "Glob" and "skill" in str(a.get("pattern", "")).lower():
            globs += 1
    rows.append({"sid": d.name, "n_read": len(reads), "n_distinct": len(set(reads)),
                 "n_glob": globs})

sk = pd.DataFrame(rows)
assert len(sk) == len(g), "扫到的份数与观测集不一致"

dist = sk.n_read.value_counts().sort_index()
fig = px.bar(x=dist.index.astype(str), y=dist.values, text=dist.values, height=360,
             title=f"每份 session 读 SKILL.md 的次数（共 {sk.n_read.sum()} 次 / {len(sk)} 份）",
             labels={"x": "读取次数", "y": "session 数"})
fig.update_traces(textposition="outside", cliponaxis=False)
fig.show()

print(f"均值 {sk.n_read.mean():.2f}，中位 {sk.n_read.median():.0f}，"
      f"最多 {sk.n_read.max()} 次")
print(f"读了同一个文件不止一次的 session：{(sk.n_read > sk.n_distinct).sum()} 份")
print(f"用 Glob 找过 skill 的 session：{(sk.n_glob > 0).sum()} 份 "
      f"（{(sk.n_glob > 0).mean():.0%}），共 {sk.n_glob.sum()} 次")
print()
print("被读到的 SKILL.md（全组合计）：")
for path, c in paths.most_common():
    print(f"  {c:5d}  {path}")

## 5. 从 session 轨迹抽取 workflow 分叉

前面统计的是工具频率与 Skill 回读。这里直接重扫已经固定时间戳的 **同一 961 份观测集**，把每条
`tool.call` 按其参数和 step 归入可观察的 workflow 信号：文献定位、provider 核验、结构化
judgment 写入、编译、校验、校验后的修复和委派。

这不是对 agent 意图的推断：**一次工具调用只表示该路径被尝试，不表示外部查询或校验成功。**
`repair` prompt 是独立启动的 session，单列而不与主任务混淆。代表样本也不手工指定；从本笔记的
`g` 所选 session 中，按明确判据稳定地选取各分叉类中步数最短的一条。

In [ ]:
from pathlib import Path

# `g` 是本笔记第 1 节选定的 961 份 session；不能在这里另选一批样本。
# 保存简短的调用明细，既足够展示代表轨迹，也不把整份 wire 载入 DataFrame。
def _brief_arg(name, args):
    if name == "Bash":
        command = str(args.get("command", ""))
        if "apply_agent_judgment" in command:
            return "apply_agent_judgment.py"
        if "validate_layer4_outputs" in command:
            return "validate_layer4_outputs.py"
        return command.splitlines()[0][:90]
    if "path" in args:
        return Path(str(args["path"])).name
    if "query" in args:
        return str(args["query"]).replace("\n", " ")[:90]
    if "repo_id" in args:
        return str(args["repo_id"])[:90]
    if "url" in args:
        return str(args["url"])[:90]
    return ", ".join(sorted(args))[:90]


def _trace_session(session_dir):
    prompt = ""
    calls = []
    for line in (session_dir / "agents/main/wire.jsonl").open(encoding="utf-8", errors="replace"):
        record = json.loads(line)
        if record.get("type") == "turn.prompt" and not prompt:
            prompt = wire._as_text(record.get("input"))
            continue
        if record.get("type") != "context.append_loop_event":
            continue
        event = record.get("event") or {}
        if event.get("type") != "tool.call":
            continue
        args = event.get("args") or {}
        calls.append({
            "step": event.get("step"),
            "tool": event.get("name") or "?",
            "detail": _brief_arg(event.get("name") or "?", args),
            "args": args,
        })

    def named(name):
        return [c for c in calls if c["tool"] == name]

    bash = [str(c["args"].get("command", "")) for c in named("Bash")]
    validate_steps = [c["step"] for c in calls
                      if c["tool"] == "Bash"
                      and "validate_layer4_outputs" in str(c["args"].get("command", ""))]
    first_validate = min((s for s in validate_steps if s is not None), default=None)
    edits_after_validate = any(
        c["tool"] == "Edit" and first_validate is not None and c["step"] is not None
        and c["step"] > first_validate
        for c in calls
    )
    tool_names = [c["tool"] for c in calls]
    return {
        "sid": session_dir.name,
        "repair_prompt": "agent_repair_prompt.md" in prompt,
        "n_steps_observed": len({c["step"] for c in calls if c["step"] is not None}),
        "n_calls": len(calls),
        "arxiv_calls": sum(t.startswith("mcp__arxiv") for t in tool_names),
        "github_calls": sum(t.startswith("mcp__github") for t in tool_names),
        "hf_calls": sum(t.startswith("mcp__hf") for t in tool_names),
        "delegated": any(t in {"Agent", "AgentSwarm"} for t in tool_names),
        "judgment_write": any(
            c["tool"] == "Write"
            and str(c["args"].get("path", "")).endswith("agent_judgment.json")
            for c in calls
        ),
        "has_edit": "Edit" in tool_names,
        "apply_called": any("apply_agent_judgment" in command for command in bash),
        "validate_called": bool(validate_steps),
        "edit_after_validate": edits_after_validate,
        "calls": calls,
    }


traces = [_trace_session(d) for d in wire.session_dirs(root=CORPUS)]
call_detail = {row["sid"]: row.pop("calls") for row in traces}
tr = pd.DataFrame(traces).set_index("sid").sort_index()

# 观测对象与第 1 节完全一致；这条断言防止从原始全量语料或其他 n0 副本误取样本。
assert set(tr.index) == set(g.sid), "轨迹扫描的 session 集合与 g 不一致"
assert len(tr) == 961

main = tr.loc[~tr.repair_prompt].copy()
main["compiled_and_validated"] = (
    main.judgment_write & main.apply_called & main.validate_called
)


def _branch(row):
    if row.edit_after_validate:
        return "validator 后定点修复"
    if row.delegated:
        return "委派 / 并行探索"
    if row.arxiv_calls >= 2:
        return "文献定位重试"
    if row.github_calls == 0:
        return "未走 GitHub MCP 核验"
    return "常规主干"


main["branch"] = main.apply(_branch, axis=1)
print(f"主任务 {len(main)} 份；独立 repair prompt {tr.repair_prompt.sum()} 份")
print("✓ 解析范围与本笔记既有观测集一致")


### 5.1 主干与分叉的可观察覆盖率

`apply` 与 `validate` 的出现说明 session 进入了既有的结构化产物链；它们不是“校验成功率”。
`Edit` 又分为普通编写阶段的 edit 与首次 validator 调用之后的 edit，后者才作为 validator 驱动 repair 的保守代理。

In [ ]:
coverage = pd.DataFrame({
    "session 数": {
        "arXiv MCP": (main.arxiv_calls > 0).sum(),
        "GitHub MCP": (main.github_calls > 0).sum(),
        "HF MCP": (main.hf_calls > 0).sum(),
        "写 agent_judgment.json": main.judgment_write.sum(),
        "调用 apply": main.apply_called.sum(),
        "调用 validate": main.validate_called.sum(),
        "任意 Edit": main.has_edit.sum(),
        "validator 后 Edit": main.edit_after_validate.sum(),
        "Agent / AgentSwarm": main.delegated.sum(),
    }
})
coverage["覆盖率"] = (coverage["session 数"] / len(main) * 100).round(1)
coverage


### 5.2 从同一观测集稳定选取代表轨迹

下表的类是**调用模式**，不是对论文内容的标签。优先级是：validator 后修复、委派、文献重试、未走
GitHub MCP、常规主干。每类只从已观察到 `Write → apply → validate` 三类动作的主任务中选步数最短、
session id 次序最小的一条；这样重新运行时不会依赖人工挑选。

In [ ]:
branch_order = [
    "常规主干",
    "文献定位重试",
    "未走 GitHub MCP 核验",
    "validator 后定点修复",
    "委派 / 并行探索",
]

branch_summary = (main.branch.value_counts()
                  .reindex(branch_order, fill_value=0)
                  .rename_axis("调用模式")
                  .rename("session 数")
                  .to_frame())
branch_summary["覆盖率"] = (branch_summary["session 数"] / len(main) * 100).round(1)
display(branch_summary)

eligible = (main[main.compiled_and_validated]
            .sort_values(["branch", "n_steps_observed", "n_calls"], kind="stable"))
examples = (eligible.reset_index()
            .sort_values(["n_steps_observed", "sid"], kind="stable")
            .groupby("branch", sort=False, as_index=False)
            .first()
            .set_index("branch")
            .reindex(branch_order)
            .dropna(subset=["sid"])
            .reset_index())

examples = examples[["branch", "sid", "n_steps_observed", "n_calls",
                     "arxiv_calls", "github_calls", "hf_calls", "has_edit", "delegated"]]
display(examples)

# 展开上述自动选择的少量 session；细节来自 wire 的真实 tool.call 参数，不读取其他 session。
example_rows = []
for ex in examples.itertuples(index=False):
    for call in call_detail[ex.sid]:
        example_rows.append({
            "调用模式": ex.branch,
            "sid": ex.sid,
            "step": call["step"],
            "tool": call["tool"],
            "参数摘要": call["detail"],
        })
example_calls = pd.DataFrame(example_rows)
example_calls


### 5.3 可归约的 Paper-for-Agents workflow 模式

从调用轨迹能保守地读出一个混合 workflow，而不是把每条 session 视作不可分割的 ReAct 样本：

```text
固定任务 / Skill / 已有输入
  → 文献定位（命中 | 重试 | 未匹配）
  → 候选资源按 provider 核验（GitHub | arXiv | HF | 其他 URL）
  → 生成 agent_judgment
  → apply：生成结构化输出
  → validate
      ├─ pass → 发布 paper-local artifacts
      ├─ 结构或证据错误 → 定点 repair / re-verification → apply → validate
      └─ 执行能力不足或异常 → agent / 人工 fallback
```

对新的 Paper-for-Agents workload，这给出可编译主干和动态尾部的边界：文献定位、provider 核验、
`apply`、`validate`、有界 repair 都有明确输入输出契约；`Agent` / `AgentSwarm`、暂停恢复和工具能力缺失
是 execution/harness 异常信号，不能直接误当作业务 DAG 的必经节点。

这仍是探索性轨迹探测：本节没有检查 tool result、没有重新裁定某次外部核验是否成功，也没有以此估计
CachePlan 的性能收益。若这些统计成为正文证据，应将本节的事件表和 outcome 判定固化为 `data/processed/`
产物及相应脚本。